In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Baca data
df = pd.read_csv("Social_media_impact_on_life.csv")

# 2. Buang kolom yang tidak dipakai
df = df.drop(columns=["Student_ID", "Overall_Impact"])

# 3. Cek duplikat
print("Baris duplikat:", df.duplicated().sum())

# 4. Ubah Late_Night_Usage (benar/salah) jadi 1/0
df["Late_Night_Usage"] = df["Late_Night_Usage"].astype(int)

# 5. Ubah kolom teks jadi angka
df = pd.get_dummies(df, drop_first=True)

# 6. Pisahkan fitur (X) dan target (y)
X = df.drop(columns=["Sleep_Quality_Score"])
y = df["Sleep_Quality_Score"]

# 7. Bagi data: 80% latih, 20% uji
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


Baris duplikat: 0


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error

# --- Rapikan data ---
df = pd.read_csv("Social_media_impact_on_life.csv")
df = df.drop(columns=["Student_ID", "Overall_Impact"])
df["Late_Night_Usage"] = df["Late_Night_Usage"].astype(int)
df = pd.get_dummies(df, drop_first=True)

X = df.drop(columns=["Sleep_Quality_Score"])
y = df["Sleep_Quality_Score"]

X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Isi sel kosong (median dari data latih saja) ---
median_latih = X_latih.median()
X_latih = X_latih.fillna(median_latih).astype(float)
X_uji = X_uji.fillna(median_latih).astype(float)

# --- Pemeriksaan: harus 0 sebelum lanjut ---
print("Kosong di latih:", X_latih.isna().sum().sum())
print("Kosong di uji  :", X_uji.isna().sum().sum())
assert X_latih.isna().sum().sum() == 0, "Masih ada sel kosong di data latih!"

# --- Model ---
daftar_model = {
    "Tebakan asal (kelas terbanyak)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

for nama, model in daftar_model.items():
    model.fit(X_latih, y_latih)
    tebakan = model.predict(X_uji)
    akurasi = accuracy_score(y_uji, tebakan)
    meleset = mean_absolute_error(y_uji, tebakan)
    print(f"{nama}: akurasi={akurasi:.2f}, rata-rata meleset={meleset:.2f} poin")

Kosong di latih: 0
Kosong di uji  : 0
Tebakan asal (kelas terbanyak): akurasi=0.34, rata-rata meleset=0.88 poin
Logistic Regression: akurasi=0.60, rata-rata meleset=0.42 poin
Random Forest: akurasi=0.58, rata-rata meleset=0.45 poin


In [ ]:
skenario = {
    "Semua fitur": [],
    "Tanpa jam tidur": ["Sleep_Duration_Hours"],
    "Tanpa jam tidur, kesehatan mental, stres, IPK": [
        "Sleep_Duration_Hours", "Mental_Health_Index",
        "Perceived_Stress_Score", "Academic_Performance_GPA",
    ],
}

for judul, buang in skenario.items():
    Xl = X_latih.drop(columns=buang)
    Xu = X_uji.drop(columns=buang)
    for nama, model in {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    }.items():
        model.fit(Xl, y_latih)
        t = model.predict(Xu)
        print(f"{judul} | {nama}: akurasi={accuracy_score(y_uji, t):.2f}, "
              f"meleset={mean_absolute_error(y_uji, t):.2f}")

In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.inspection import permutation_importance

buang = ["Sleep_Duration_Hours", "Mental_Health_Index",
         "Perceived_Stress_Score", "Academic_Performance_GPA"]

# (a) k-fold pada fitur perilaku saja
X_perilaku = X.drop(columns=buang).astype(float)
print("Kosong:", X_perilaku.isna().sum().sum())   # harus 0

model_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lipat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skor = cross_val_score(model_lr, X_perilaku, y, cv=lipat, scoring="accuracy")
print("Akurasi per lipatan:", skor.round(2))
print(f"Rata-rata: {skor.mean():.2f}  (variasi: ±{skor.std():.2f})")

# (b) kolom mana yang paling berpengaruh
Xl = X_latih.drop(columns=buang)
Xu = X_uji.drop(columns=buang)
model_lr.fit(Xl, y_latih)
hasil = permutation_importance(model_lr, Xu, y_uji, n_repeats=10, random_state=42)
urut = pd.Series(hasil.importances_mean, index=Xu.columns).sort_values(ascending=False)
print(urut.head(8).round(3))

Kosong: 0
Akurasi per lipatan: [0.54 0.53 0.51 0.51 0.49]
Rata-rata: 0.52  (variasi: ±0.02)
Daily_Usage_Hours                         0.211
Academic_Level_Undergraduate              0.006
Gender_Non-Binary                         0.005
Primary_Platform_Reddit                   0.003
Social_Comparison_Frequency_Frequently    0.003
Social_Comparison_Frequency_Sometimes     0.002
Primary_Platform_LinkedIn                 0.002
Device_Type_Tablet                        0.002
dtype: float64


In [9]:
from sklearn.model_selection import cross_val_predict

# 1. Hanya satu fitur: apakah hasilnya hampir sama dengan 0,52?
X_satu = X_perilaku[["Daily_Usage_Hours"]]
skor_satu = cross_val_score(model_lr, X_satu, y, cv=lipat, scoring="accuracy")
print(f"Hanya jam medsos: {skor_satu.mean():.2f}")

# 2. Rata-rata skor tidur per kelompok jam pemakaian
kelompok = pd.cut(df["Daily_Usage_Hours"], bins=[0, 2, 4, 6, 8, 10, 15])
print(df.groupby(kelompok, observed=True)["Sleep_Quality_Score"].agg(["mean", "count"]).round(2))

# 3. Tebakan vs jawaban asli
tebakan = cross_val_predict(model_lr, X_perilaku, y, cv=lipat)
print(pd.crosstab(y, tebakan, rownames=["asli"], colnames=["tebakan"]))

Hanya jam medsos: 0.52
                   mean  count
Daily_Usage_Hours             
(0, 2]             3.71    178
(2, 4]             3.12   1512
(4, 6]             2.50   1452
(6, 8]             1.86    726
(8, 10]            1.35    350
(10, 15]           1.04    282
tebakan    1    2     3    4  5
asli                           
1        606  248    78    2  0
2        192  539   560   20  0
3         36  349  1060   98  0
4          2   47   457  114  1
5          0    1    57   33  0
